In [1]:
# Import libraries and define configs
import json
import random
import warnings
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import resample

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "figure.dpi": 120,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
    }
)

RANDOM_STATE = 121
N_JOBS = -1
CV_FOLDS = 5
OPERATING_THRESHOLD = 0.065424
OUTREACH_CAPACITIES = (0.01, 0.05, 0.10, 0.20)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

current_directory = Path.cwd().resolve()
project_root_candidates = [current_directory, *current_directory.parents]

PROJECT_ROOT = next(
    (
        candidate_directory
        for candidate_directory in project_root_candidates
        if (candidate_directory / "data" / "processed").exists()
        and (candidate_directory / "notebooks").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/processed and notebooks."
    )

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELING_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "modeling"
PRIVATE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "private"
INTERPRETABILITY_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "interpretability"

INTERPRETABILITY_TABLES_DIR = INTERPRETABILITY_OUTPUT_DIR / "tables"
INTERPRETABILITY_FIGURES_DIR = INTERPRETABILITY_OUTPUT_DIR / "figures"

INTERPRETABILITY_TABLES_DIR.mkdir(parents=True, exist_ok=True)
INTERPRETABILITY_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DONOR_FEATURES_PARQUET_PATH = PROCESSED_DATA_DIR / "donor_features.parquet"
DONOR_FEATURES_CSV_PATH = PROCESSED_DATA_DIR / "donor_features.csv"
FEATURE_DICTIONARY_PATH = REPORTS_DIR / "feature_dictionary.csv"

PRIMARY_PIPELINE_PATH = MODELS_DIR / "final_primary_pipeline.joblib"
BENCHMARK_PIPELINE_PATH = (
    MODELS_DIR / "historical_donor_status_benchmark_pipeline.joblib"
)

FINAL_MODEL_CONFIGURATION_PATH = (
    MODELING_OUTPUT_DIR / "final_model_configuration.json"
)
FINAL_FEATURE_LISTS_PATH = MODELING_OUTPUT_DIR / "final_feature_lists.json"
FINAL_TEST_PREDICTIONS_PATH = (
    PRIVATE_OUTPUT_DIR / "primary_donor_predictions_final_test.csv"
)
PRIMARY_TEST_METRICS_PATH = (
    MODELING_OUTPUT_DIR / "primary_final_test_metrics.csv"
)
PRIMARY_OUTREACH_RESULTS_PATH = (
    MODELING_OUTPUT_DIR / "primary_final_test_outreach_results.csv"
)
BENCHMARK_TEST_METRICS_PATH = (
    MODELING_OUTPUT_DIR / "benchmark_test_metrics.csv"
)
MODELING_ARTIFACT_MANIFEST_PATH = (
    MODELING_OUTPUT_DIR / "modeling_artifact_manifest.csv"
)

In [2]:
print(f"Project root: {PROJECT_ROOT.name}")
print(f"Random state: {RANDOM_STATE}")
print(f"Operating threshold: {OPERATING_THRESHOLD:.6f}")
print("Phase 6 interpretability setup complete.")

Project root: red-cross-donor-prediction
Random state: 121
Operating threshold: 0.065424
Phase 6 interpretability setup complete.


In [3]:
# Load models, metadata, predicts, and eval artifacts
BENCHMARK_MODEL_COMPARISON_PATH = (
    MODELING_OUTPUT_DIR / "benchmark_model_comparison.csv"
)

required_phase5_artifact_paths = {
    "Primary model pipeline": PRIMARY_PIPELINE_PATH,
    "Benchmark model pipeline": BENCHMARK_PIPELINE_PATH,
    "Final model configuration": FINAL_MODEL_CONFIGURATION_PATH,
    "Final feature lists": FINAL_FEATURE_LISTS_PATH,
    "Feature dictionary": FEATURE_DICTIONARY_PATH,
    "Primary test predictions": FINAL_TEST_PREDICTIONS_PATH,
    "Primary test metrics": PRIMARY_TEST_METRICS_PATH,
    "Primary outreach results": PRIMARY_OUTREACH_RESULTS_PATH,
    "Benchmark model comparison": BENCHMARK_MODEL_COMPARISON_PATH,
    "Benchmark test metrics": BENCHMARK_TEST_METRICS_PATH,
    "Modeling artifact manifest": MODELING_ARTIFACT_MANIFEST_PATH,
}

missing_phase5_artifacts = {
    artifact_name: artifact_path
    for artifact_name, artifact_path in required_phase5_artifact_paths.items()
    if not artifact_path.exists()
}

if missing_phase5_artifacts:
    missing_artifact_list = "\n".join(
        f"- {artifact_name}: {artifact_path.relative_to(PROJECT_ROOT)}"
        for artifact_name, artifact_path in missing_phase5_artifacts.items()
    )
    raise FileNotFoundError(
        f"Required Phase 5 artifacts were not found:\n{missing_artifact_list}"
    )

model_primary_pipeline_final = joblib.load(PRIMARY_PIPELINE_PATH)
model_benchmark_pipeline_final = joblib.load(BENCHMARK_PIPELINE_PATH)

with FINAL_MODEL_CONFIGURATION_PATH.open("r", encoding="utf-8") as file:
    configuration_final_model = json.load(file)

with FINAL_FEATURE_LISTS_PATH.open("r", encoding="utf-8") as file:
    feature_sets_final = json.load(file)

metadata_feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)
predictions_primary_final_test = pd.read_csv(FINAL_TEST_PREDICTIONS_PATH)

tracking_primary_final_test_donor_ids = (
    predictions_primary_final_test.loc[:, ["donor_unique_id"]].copy()
)

metrics_primary_final_test = pd.read_csv(PRIMARY_TEST_METRICS_PATH)
metrics_primary_outreach = pd.read_csv(PRIMARY_OUTREACH_RESULTS_PATH)
comparison_benchmark_models = pd.read_csv(BENCHMARK_MODEL_COMPARISON_PATH)
metrics_benchmark_final_test = pd.read_csv(BENCHMARK_TEST_METRICS_PATH)
manifest_modeling_artifacts = pd.read_csv(MODELING_ARTIFACT_MANIFEST_PATH)

primary_pipeline_steps = getattr(
    model_primary_pipeline_final,
    "named_steps",
    {},
)
benchmark_pipeline_steps = getattr(
    model_benchmark_pipeline_final,
    "named_steps",
    {},
)

preprocessor_primary_model_final = primary_pipeline_steps.get("preprocessor")
preprocessor_benchmark_model_final = benchmark_pipeline_steps.get("preprocessor")

if preprocessor_primary_model_final is None:
    primary_preprocessor_candidates = [
        pipeline_step
        for pipeline_step in primary_pipeline_steps.values()
        if hasattr(pipeline_step, "transformers_")
    ]

    if len(primary_preprocessor_candidates) == 1:
        preprocessor_primary_model_final = primary_preprocessor_candidates[0]
    else:
        raise KeyError(
            "The fitted primary preprocessing step could not be identified."
        )

if preprocessor_benchmark_model_final is None:
    benchmark_preprocessor_candidates = [
        pipeline_step
        for pipeline_step in benchmark_pipeline_steps.values()
        if hasattr(pipeline_step, "transformers_")
    ]

    if len(benchmark_preprocessor_candidates) == 1:
        preprocessor_benchmark_model_final = benchmark_preprocessor_candidates[0]
    else:
        raise KeyError(
            "The fitted benchmark preprocessing step could not be identified."
        )

loaded_artifact_summary = pd.DataFrame(
    [
        {
            "Artifact": "Primary model pipeline",
            "Object Type": type(model_primary_pipeline_final).__name__,
            "Rows": pd.NA,
            "Columns": pd.NA,
            "Source": PRIMARY_PIPELINE_PATH.relative_to(PROJECT_ROOT).as_posix(),
        },
        {
            "Artifact": "Benchmark model pipeline",
            "Object Type": type(model_benchmark_pipeline_final).__name__,
            "Rows": pd.NA,
            "Columns": pd.NA,
            "Source": BENCHMARK_PIPELINE_PATH.relative_to(PROJECT_ROOT).as_posix(),
        },
        {
            "Artifact": "Feature dictionary",
            "Object Type": type(metadata_feature_dictionary).__name__,
            "Rows": metadata_feature_dictionary.shape[0],
            "Columns": metadata_feature_dictionary.shape[1],
            "Source": FEATURE_DICTIONARY_PATH.relative_to(PROJECT_ROOT).as_posix(),
        },
        {
            "Artifact": "Primary final-test predictions",
            "Object Type": type(predictions_primary_final_test).__name__,
            "Rows": predictions_primary_final_test.shape[0],
            "Columns": predictions_primary_final_test.shape[1],
            "Source": FINAL_TEST_PREDICTIONS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Primary final-test metrics",
            "Object Type": type(metrics_primary_final_test).__name__,
            "Rows": metrics_primary_final_test.shape[0],
            "Columns": metrics_primary_final_test.shape[1],
            "Source": PRIMARY_TEST_METRICS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Primary outreach results",
            "Object Type": type(metrics_primary_outreach).__name__,
            "Rows": metrics_primary_outreach.shape[0],
            "Columns": metrics_primary_outreach.shape[1],
            "Source": PRIMARY_OUTREACH_RESULTS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Benchmark model comparison",
            "Object Type": type(comparison_benchmark_models).__name__,
            "Rows": comparison_benchmark_models.shape[0],
            "Columns": comparison_benchmark_models.shape[1],
            "Source": BENCHMARK_MODEL_COMPARISON_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Benchmark final-test metrics",
            "Object Type": type(metrics_benchmark_final_test).__name__,
            "Rows": metrics_benchmark_final_test.shape[0],
            "Columns": metrics_benchmark_final_test.shape[1],
            "Source": BENCHMARK_TEST_METRICS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Modeling artifact manifest",
            "Object Type": type(manifest_modeling_artifacts).__name__,
            "Rows": manifest_modeling_artifacts.shape[0],
            "Columns": manifest_modeling_artifacts.shape[1],
            "Source": MODELING_ARTIFACT_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
    ]
)

display(
    loaded_artifact_summary.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center", "padding": "8px"})
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("padding", "8px"),
                ],
            }
        ]
    )
    .format(
        {
            "Rows": "{:,.0f}",
            "Columns": "{:,.0f}",
        },
        na_rep="—",
    )
)

print(f"Primary pipeline steps: {list(primary_pipeline_steps)}")
print(f"Benchmark pipeline steps: {list(benchmark_pipeline_steps)}")
print(
    "Final-test donor IDs loaded: "
    f"{tracking_primary_final_test_donor_ids.shape[0]:,}"
)
print("Phase 5 artifacts loaded successfully.")

Artifact,Object Type,Rows,Columns,Source
Primary model pipeline,Pipeline,—,—,models/final_primary_pipeline.joblib
Benchmark model pipeline,Pipeline,—,—,models/historical_donor_status_benchmark_pipeline.joblib
Feature dictionary,DataFrame,77,7,reports/feature_dictionary.csv
Primary final-test predictions,DataFrame,"6,881",7,outputs/private/primary_donor_predictions_final_test.csv
Primary final-test metrics,DataFrame,10,2,outputs/modeling/primary_final_test_metrics.csv
Primary outreach results,DataFrame,4,8,outputs/modeling/primary_final_test_outreach_results.csv
Benchmark model comparison,DataFrame,5,20,outputs/modeling/benchmark_model_comparison.csv
Benchmark final-test metrics,DataFrame,10,2,outputs/modeling/benchmark_test_metrics.csv
Modeling artifact manifest,DataFrame,12,5,outputs/modeling/modeling_artifact_manifest.csv


Primary pipeline steps: ['preprocessor', 'classifier']
Benchmark pipeline steps: ['preprocessor', 'classifier']
Final-test donor IDs loaded: 6,881
Phase 5 artifacts loaded successfully.


In [4]:
# Inspect artifact structures before validation
print("FINAL MODEL CONFIGURATION")
print(json.dumps(configuration_final_model, indent=2))

print("\nFINAL FEATURE LISTS")
print(json.dumps(feature_sets_final, indent=2))

print("\nPRIMARY PIPELINE INPUT FEATURES")
print(list(model_primary_pipeline_final.feature_names_in_))

print("\nPRIMARY CLASSIFIER PARAMETERS")
primary_classifier_parameters = primary_pipeline_steps[
    "classifier"
].get_params()

print(
    {
        parameter_name: primary_classifier_parameters.get(parameter_name)
        for parameter_name in [
            "n_estimators",
            "max_depth",
            "min_samples_split",
            "min_samples_leaf",
            "max_features",
            "class_weight",
            "random_state",
        ]
    }
)

print("\nFINAL-TEST PREDICTION COLUMNS")
print(predictions_primary_final_test.columns.tolist())

print("\nPRIMARY TEST METRIC COLUMNS")
print(metrics_primary_final_test.columns.tolist())

print("\nPRIMARY OUTREACH RESULT COLUMNS")
print(metrics_primary_outreach.columns.tolist())

display(metrics_primary_final_test)
display(metrics_primary_outreach)

FINAL MODEL CONFIGURATION
{
  "primary": {
    "target": "target_current_fiscal_year_donor_flag",
    "model": "RF Moderate",
    "configuration_id": "rf_depth_14",
    "probability_version": "Uncalibrated",
    "feature_set": "Aggregate RFM",
    "source_feature_count": 8,
    "operating_threshold": 0.06542368672155528,
    "random_state": 121,
    "test_size": 0.2,
    "cv_folds": 5,
    "primary_metric": "average_precision",
    "outreach_capacities": [
      0.01,
      0.05,
      0.1,
      0.2
    ]
  },
  "benchmark": {
    "target": "donor_indicator_flag",
    "purpose": "Historical donor status cohort comparison only",
    "model": "Decision Tree",
    "predictor_count": 16,
    "classification_threshold": 0.5,
    "random_state": 122,
    "test_size": 0.2,
    "cv_folds": 5,
    "primary_metric": "roc_auc"
  }
}

FINAL FEATURE LISTS
{
  "primary_final_features": [
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_donation_f

,metric,value
0,pr_auc_average_precision,0.07
1,roc_auc,0.52
2,recall,0.10
3,precision,0.07
4,f1_score,0.08
5,specificity,0.92
6,balanced_accuracy,0.51
7,predicted_positive_rate,0.08
8,brier_score,0.05
9,probability_threshold,0.07


,campaign_capacity,individuals_contacted,actual_donors_captured,recall_among_all_donors,precision_selected_group,lift_over_random,expected_random_donors,additional_donors_vs_random
0,0.01,69,13,0.03,0.19,3.40,3.82,9.18
1,0.05,345,23,0.06,0.07,1.20,19.10,3.90
2,0.10,689,46,0.12,0.07,1.21,38.15,7.85
3,0.20,1377,84,0.22,0.06,1.10,76.24,7.76


In [5]:
# Validate Phase 5 decisions
expected_primary_decisions = {
    "target": "target_current_fiscal_year_donor_flag",
    "model": "RF Moderate",
    "configuration_id": "rf_depth_14",
    "probability_version": "Uncalibrated",
    "feature_set": "Aggregate RFM",
    "source_feature_count": 8,
    "transformed_feature_count": 14,
    "operating_threshold": 0.06542368672155528,
    "random_state": 121,
    "test_size": 0.20,
    "cv_folds": 5,
    "primary_metric": "average_precision",
    "outreach_capacities": (0.01, 0.05, 0.10, 0.20),
    "total_records": 34_403,
    "development_records": 27_522,
    "final_test_records": 6_881,
    "total_positives": 1_904,
    "development_positives": 1_523,
    "final_test_positives": 381,
}

expected_primary_features = [
    "feature_past_5yr_total_donation",
    "feature_past_5yr_average_donation",
    "feature_past_5yr_donation_frequency_rate",
    "feature_years_since_last_donation_past_5yr",
    "feature_past_5yr_max_donation",
    "donor_age",
    "feature_gender_identity",
    "feature_preferred_address_type",
]

expected_classifier_parameters = {
    "n_estimators": 400,
    "max_depth": 14,
    "min_samples_split": 20,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "class_weight": None,
    "random_state": 121,
}

if DONOR_FEATURES_PARQUET_PATH.exists():
    data_modeling_full = pd.read_parquet(DONOR_FEATURES_PARQUET_PATH)
    modeling_data_source_path = DONOR_FEATURES_PARQUET_PATH
elif DONOR_FEATURES_CSV_PATH.exists():
    data_modeling_full = pd.read_csv(DONOR_FEATURES_CSV_PATH)
    modeling_data_source_path = DONOR_FEATURES_CSV_PATH
else:
    raise FileNotFoundError(
        "Neither donor_features.parquet nor donor_features.csv was found."
    )

required_modeling_columns = {
    "donor_unique_id",
    expected_primary_decisions["target"],
}

missing_modeling_columns = required_modeling_columns.difference(
    data_modeling_full.columns
)

if missing_modeling_columns:
    raise KeyError(
        "Required modeling columns are missing: "
        + ", ".join(sorted(missing_modeling_columns))
    )

configuration_primary_saved = configuration_final_model["primary"]
configuration_benchmark_saved = configuration_final_model["benchmark"]

features_primary_saved = feature_sets_final["primary_final_features"]
features_aggregate_rfm_saved = feature_sets_final[
    "primary_feature_set_variants"
]["Safe Aggregate RFM Set"]

features_primary_pipeline = list(
    model_primary_pipeline_final.feature_names_in_
)

features_primary_transformed = list(
    preprocessor_primary_model_final.get_feature_names_out()
)

classifier_primary_saved = primary_pipeline_steps["classifier"]
classifier_parameters_observed = {
    parameter_name: classifier_primary_saved.get_params()[parameter_name]
    for parameter_name in expected_classifier_parameters
}

metadata_feature_name_column = next(
    (
        column_name
        for column_name in metadata_feature_dictionary.columns
        if set(expected_primary_features).issubset(
            set(metadata_feature_dictionary[column_name].astype(str))
        )
    ),
    None,
)

test_donor_ids_observed = set(
    predictions_primary_final_test["donor_unique_id"]
)

test_ids_unique_nonmissing = (
    predictions_primary_final_test["donor_unique_id"].notna().all()
    and predictions_primary_final_test["donor_unique_id"].is_unique
)

modeling_ids_unique_nonmissing = (
    data_modeling_full["donor_unique_id"].notna().all()
    and data_modeling_full["donor_unique_id"].is_unique
)

test_ids_exist_in_modeling_data = test_donor_ids_observed.issubset(
    set(data_modeling_full["donor_unique_id"])
)

data_development_reconstructed = data_modeling_full.loc[
    ~data_modeling_full["donor_unique_id"].isin(test_donor_ids_observed)
].copy()

test_target_comparison = predictions_primary_final_test[
    ["donor_unique_id", "target_actual_primary_donor_flag"]
].merge(
    data_modeling_full[
        [
            "donor_unique_id",
            expected_primary_decisions["target"],
        ]
    ],
    on="donor_unique_id",
    how="left",
    validate="one_to_one",
)

test_targets_match_source = (
    test_target_comparison[
        expected_primary_decisions["target"]
    ].notna().all()
    and np.array_equal(
        test_target_comparison[
            "target_actual_primary_donor_flag"
        ].astype(int),
        test_target_comparison[
            expected_primary_decisions["target"]
        ].astype(int),
    )
)

total_record_count_observed = data_modeling_full.shape[0]
development_record_count_observed = data_development_reconstructed.shape[0]
test_record_count_observed = predictions_primary_final_test.shape[0]

total_positive_count_observed = int(
    data_modeling_full[expected_primary_decisions["target"]].sum()
)
development_positive_count_observed = int(
    data_development_reconstructed[
        expected_primary_decisions["target"]
    ].sum()
)
test_positive_count_observed = int(
    predictions_primary_final_test[
        "target_actual_primary_donor_flag"
    ].sum()
)

operating_threshold_saved = float(
    configuration_primary_saved["operating_threshold"]
)
outreach_capacities_saved = tuple(
    float(capacity)
    for capacity in configuration_primary_saved["outreach_capacities"]
)

probabilities_primary_test = predictions_primary_final_test[
    "prob_primary_donor"
].astype(float)

target_primary_test = predictions_primary_final_test[
    "target_actual_primary_donor_flag"
].astype(int)

predictions_primary_test_saved = predictions_primary_final_test[
    "pred_primary_donor_flag"
].astype(int)

predictions_primary_test_recalculated = (
    probabilities_primary_test >= operating_threshold_saved
).astype(int)

threshold_predictions_match = np.array_equal(
    predictions_primary_test_saved,
    predictions_primary_test_recalculated,
)

true_negative, false_positive, false_negative, true_positive = (
    confusion_matrix(
        target_primary_test,
        predictions_primary_test_recalculated,
        labels=[0, 1],
    ).ravel()
)

specificity_primary_test = (
    true_negative / (true_negative + false_positive)
)

recall_primary_test = recall_score(
    target_primary_test,
    predictions_primary_test_recalculated,
    zero_division=0,
)

calculated_primary_test_metrics = {
    "pr_auc_average_precision": average_precision_score(
        target_primary_test,
        probabilities_primary_test,
    ),
    "roc_auc": roc_auc_score(
        target_primary_test,
        probabilities_primary_test,
    ),
    "recall": recall_primary_test,
    "precision": precision_score(
        target_primary_test,
        predictions_primary_test_recalculated,
        zero_division=0,
    ),
    "f1_score": f1_score(
        target_primary_test,
        predictions_primary_test_recalculated,
        zero_division=0,
    ),
    "specificity": specificity_primary_test,
    "balanced_accuracy": (
        recall_primary_test + specificity_primary_test
    )
    / 2,
    "predicted_positive_rate": float(
        predictions_primary_test_recalculated.mean()
    ),
    "brier_score": float(
        np.mean(
            (
                probabilities_primary_test
                - target_primary_test
            )
            ** 2
        )
    ),
    "probability_threshold": operating_threshold_saved,
}

saved_primary_test_metrics = (
    metrics_primary_final_test
    .set_index("metric")["value"]
    .astype(float)
    .to_dict()
)

required_metric_names = set(calculated_primary_test_metrics)
saved_metrics_complete = required_metric_names.issubset(
    saved_primary_test_metrics
)

test_metrics_reproduce = (
    saved_metrics_complete
    and all(
        np.isclose(
            calculated_primary_test_metrics[metric_name],
            saved_primary_test_metrics[metric_name],
            rtol=1e-7,
            atol=1e-10,
        )
        for metric_name in required_metric_names
    )
)

test_positive_rate_observed = float(target_primary_test.mean())

recalculated_outreach_records = []

for campaign_capacity in outreach_capacities_saved:
    individuals_contacted = int(
        np.ceil(test_record_count_observed * campaign_capacity)
    )

    selected_outreach_records = (
        predictions_primary_final_test
        .nsmallest(individuals_contacted, "donor_rank")
    )

    actual_donors_captured = int(
        selected_outreach_records[
            "target_actual_primary_donor_flag"
        ].sum()
    )

    precision_selected_group = (
        actual_donors_captured / individuals_contacted
    )
    recall_among_all_donors = (
        actual_donors_captured / test_positive_count_observed
    )
    expected_random_donors = (
        individuals_contacted * test_positive_rate_observed
    )

    recalculated_outreach_records.append(
        {
            "campaign_capacity": campaign_capacity,
            "individuals_contacted": individuals_contacted,
            "actual_donors_captured": actual_donors_captured,
            "recall_among_all_donors": recall_among_all_donors,
            "precision_selected_group": precision_selected_group,
            "lift_over_random": (
                precision_selected_group
                / test_positive_rate_observed
            ),
            "expected_random_donors": expected_random_donors,
            "additional_donors_vs_random": (
                actual_donors_captured
                - expected_random_donors
            ),
        }
    )

metrics_primary_outreach_recalculated = pd.DataFrame(
    recalculated_outreach_records
)

outreach_comparison = metrics_primary_outreach.merge(
    metrics_primary_outreach_recalculated,
    on="campaign_capacity",
    how="outer",
    suffixes=("_saved", "_recalculated"),
    validate="one_to_one",
)

outreach_metric_names = [
    "individuals_contacted",
    "actual_donors_captured",
    "recall_among_all_donors",
    "precision_selected_group",
    "lift_over_random",
    "expected_random_donors",
    "additional_donors_vs_random",
]

outreach_metrics_reproduce = (
    outreach_comparison.shape[0]
    == len(outreach_capacities_saved)
    and all(
        np.allclose(
            outreach_comparison[
                f"{metric_name}_saved"
            ].astype(float),
            outreach_comparison[
                f"{metric_name}_recalculated"
            ].astype(float),
            rtol=1e-7,
            atol=1e-10,
        )
        for metric_name in outreach_metric_names
    )
)

selected_outreach_flag_recalculated = (
    predictions_primary_test_recalculated.copy()
)

selected_outreach_flags_match = np.array_equal(
    selected_outreach_flag_recalculated.to_numpy(),
    predictions_primary_final_test[
        "selected_outreach_flag"
    ].astype(int).to_numpy(),
)

validation_records = []


def add_validation(
    decision,
    expected,
    observed,
    passed,
):
    validation_records.append(
        {
            "Decision": decision,
            "Expected": str(expected),
            "Observed": str(observed),
            "Passed": bool(passed),
        }
    )


add_validation(
    "Primary target",
    expected_primary_decisions["target"],
    configuration_primary_saved["target"],
    configuration_primary_saved["target"]
    == expected_primary_decisions["target"],
)

add_validation(
    "Selected model label",
    expected_primary_decisions["model"],
    configuration_primary_saved["model"],
    configuration_primary_saved["model"]
    == expected_primary_decisions["model"],
)

add_validation(
    "Selected model class",
    "RandomForestClassifier",
    type(classifier_primary_saved).__name__,
    type(classifier_primary_saved).__name__
    == "RandomForestClassifier",
)

add_validation(
    "Configuration ID",
    expected_primary_decisions["configuration_id"],
    configuration_primary_saved["configuration_id"],
    configuration_primary_saved["configuration_id"]
    == expected_primary_decisions["configuration_id"],
)

add_validation(
    "Probability version",
    expected_primary_decisions["probability_version"],
    configuration_primary_saved["probability_version"],
    configuration_primary_saved["probability_version"]
    == expected_primary_decisions["probability_version"],
)

add_validation(
    "Selected feature set",
    expected_primary_decisions["feature_set"],
    configuration_primary_saved["feature_set"],
    configuration_primary_saved["feature_set"]
    == expected_primary_decisions["feature_set"],
)

add_validation(
    "Source feature count",
    expected_primary_decisions["source_feature_count"],
    len(features_primary_pipeline),
    (
        configuration_primary_saved["source_feature_count"]
        == expected_primary_decisions["source_feature_count"]
        == len(features_primary_saved)
        == len(features_aggregate_rfm_saved)
        == len(features_primary_pipeline)
    ),
)

add_validation(
    "Source feature identities and order",
    "Exact eight-feature match",
    (
        "Exact match"
        if features_primary_pipeline
        == expected_primary_features
        else "Mismatch"
    ),
    (
        features_primary_saved
        == expected_primary_features
        and features_aggregate_rfm_saved
        == expected_primary_features
        and features_primary_pipeline
        == expected_primary_features
    ),
)

add_validation(
    "Transformed feature count",
    expected_primary_decisions[
        "transformed_feature_count"
    ],
    len(features_primary_transformed),
    len(features_primary_transformed)
    == expected_primary_decisions[
        "transformed_feature_count"
    ],
)

add_validation(
    "Feature dictionary coverage",
    "All eight source features",
    (
        f"All found in {metadata_feature_name_column}"
        if metadata_feature_name_column is not None
        else "Incomplete"
    ),
    metadata_feature_name_column is not None,
)

add_validation(
    "Classifier parameters",
    expected_classifier_parameters,
    classifier_parameters_observed,
    classifier_parameters_observed
    == expected_classifier_parameters,
)

add_validation(
    "Random state",
    expected_primary_decisions["random_state"],
    configuration_primary_saved["random_state"],
    (
        configuration_primary_saved["random_state"]
        == expected_primary_decisions["random_state"]
        == classifier_primary_saved.random_state
    ),
)

add_validation(
    "Test-size setting",
    f"{expected_primary_decisions['test_size']:.0%}",
    f"{configuration_primary_saved['test_size']:.0%}",
    np.isclose(
        configuration_primary_saved["test_size"],
        expected_primary_decisions["test_size"],
    ),
)

add_validation(
    "Cross-validation folds",
    expected_primary_decisions["cv_folds"],
    configuration_primary_saved["cv_folds"],
    configuration_primary_saved["cv_folds"]
    == expected_primary_decisions["cv_folds"],
)

add_validation(
    "Primary selection metric",
    expected_primary_decisions["primary_metric"],
    configuration_primary_saved["primary_metric"],
    configuration_primary_saved["primary_metric"]
    == expected_primary_decisions["primary_metric"],
)

add_validation(
    "Modeling donor IDs",
    "Unique and nonmissing",
    (
        "Unique and nonmissing"
        if modeling_ids_unique_nonmissing
        else "Invalid"
    ),
    modeling_ids_unique_nonmissing,
)

add_validation(
    "Final-test donor IDs",
    "Unique, nonmissing, and present in source data",
    (
        "Validated"
        if (
            test_ids_unique_nonmissing
            and test_ids_exist_in_modeling_data
        )
        else "Invalid"
    ),
    (
        test_ids_unique_nonmissing
        and test_ids_exist_in_modeling_data
    ),
)

add_validation(
    "Final-test target alignment",
    "Exact match with modeling dataset",
    (
        "Exact match"
        if test_targets_match_source
        else "Mismatch"
    ),
    test_targets_match_source,
)

add_validation(
    "Total records",
    f"{expected_primary_decisions['total_records']:,}",
    f"{total_record_count_observed:,}",
    total_record_count_observed
    == expected_primary_decisions["total_records"],
)

add_validation(
    "Development records",
    f"{expected_primary_decisions['development_records']:,}",
    f"{development_record_count_observed:,}",
    development_record_count_observed
    == expected_primary_decisions["development_records"],
)

add_validation(
    "Final-test records",
    f"{expected_primary_decisions['final_test_records']:,}",
    f"{test_record_count_observed:,}",
    test_record_count_observed
    == expected_primary_decisions["final_test_records"],
)

add_validation(
    "Total positive targets",
    f"{expected_primary_decisions['total_positives']:,}",
    f"{total_positive_count_observed:,}",
    total_positive_count_observed
    == expected_primary_decisions["total_positives"],
)

add_validation(
    "Development positive targets",
    f"{expected_primary_decisions['development_positives']:,}",
    f"{development_positive_count_observed:,}",
    development_positive_count_observed
    == expected_primary_decisions["development_positives"],
)

add_validation(
    "Final-test positive targets",
    f"{expected_primary_decisions['final_test_positives']:,}",
    f"{test_positive_count_observed:,}",
    test_positive_count_observed
    == expected_primary_decisions["final_test_positives"],
)

add_validation(
    "Operating threshold",
    f"{expected_primary_decisions['operating_threshold']:.15f}",
    f"{operating_threshold_saved:.15f}",
    np.isclose(
        operating_threshold_saved,
        expected_primary_decisions["operating_threshold"],
        rtol=0,
        atol=1e-15,
    ),
)

add_validation(
    "Threshold predictions",
    "Exact reproduction",
    (
        "Exact reproduction"
        if threshold_predictions_match
        else "Mismatch"
    ),
    threshold_predictions_match,
)

add_validation(
    "Outreach capacities",
    ", ".join(
        f"{capacity:.0%}"
        for capacity in expected_primary_decisions[
            "outreach_capacities"
        ]
    ),
    ", ".join(
        f"{capacity:.0%}"
        for capacity in outreach_capacities_saved
    ),
    (
        outreach_capacities_saved
        == expected_primary_decisions[
            "outreach_capacities"
        ]
        and tuple(
            metrics_primary_outreach[
                "campaign_capacity"
            ].astype(float)
        )
        == expected_primary_decisions[
            "outreach_capacities"
        ]
    ),
)

add_validation(
    "Primary test metrics",
    "Reproduced from saved predictions",
    (
        "All metrics reproduced"
        if test_metrics_reproduce
        else "Mismatch"
    ),
    test_metrics_reproduce,
)

add_validation(
    "Outreach results",
    "Reproduced at all four capacities",
    (
        "All results reproduced"
        if outreach_metrics_reproduce
        else "Mismatch"
    ),
    outreach_metrics_reproduce,
)

add_validation(
    "Threshold-based outreach flags",
    "Match operating-threshold classifications",
    (
        "Exact reproduction"
        if selected_outreach_flags_match
        else "Mismatch"
    ),
    selected_outreach_flags_match,
)

add_validation(
    "Separate benchmark target",
    "donor_indicator_flag",
    configuration_benchmark_saved["target"],
    (
        configuration_benchmark_saved["target"]
        == "donor_indicator_flag"
        and configuration_benchmark_saved["target"]
        != configuration_primary_saved["target"]
    ),
)

validation_phase5_frozen_decisions = pd.DataFrame(
    validation_records
)

validation_phase5_frozen_decisions["Status"] = np.where(
    validation_phase5_frozen_decisions["Passed"],
    "PASS",
    "FAIL",
)

display(
    validation_phase5_frozen_decisions[
        ["Decision", "Expected", "Observed", "Status"]
    ].style
    .hide(axis="index")
    .set_properties(
        **{
            "text-align": "center",
            "padding": "8px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("padding", "8px"),
                ],
            }
        ]
    )
    .format(
        {
            "Decision": "{}",
            "Expected": "{}",
            "Observed": "{}",
            "Status": "{}",
        }
    )
)

passed_validation_count = int(
    validation_phase5_frozen_decisions["Passed"].sum()
)
total_validation_count = validation_phase5_frozen_decisions.shape[0]

if passed_validation_count != total_validation_count:
    failed_validation_names = validation_phase5_frozen_decisions.loc[
        ~validation_phase5_frozen_decisions["Passed"],
        "Decision",
    ].tolist()

    raise AssertionError(
        "Frozen Phase 5 validation failed for: "
        + ", ".join(failed_validation_names)
    )

OPERATING_THRESHOLD = operating_threshold_saved
OUTREACH_CAPACITIES = outreach_capacities_saved

print(
    "Frozen Phase 5 decisions independently validated: "
    f"{passed_validation_count}/{total_validation_count} checks passed."
)
print(
    "Modeling data source: "
    f"{modeling_data_source_path.relative_to(PROJECT_ROOT)}"
)
print(
    "Exact operating threshold retained for Phase 6: "
    f"{OPERATING_THRESHOLD:.15f}"
)

Decision,Expected,Observed,Status
Primary target,target_current_fiscal_year_donor_flag,target_current_fiscal_year_donor_flag,PASS
Selected model label,RF Moderate,RF Moderate,PASS
Selected model class,RandomForestClassifier,RandomForestClassifier,PASS
Configuration ID,rf_depth_14,rf_depth_14,PASS
Probability version,Uncalibrated,Uncalibrated,PASS
Selected feature set,Aggregate RFM,Aggregate RFM,PASS
Source feature count,8,8,PASS
Source feature identities and order,Exact eight-feature match,Exact match,PASS
Transformed feature count,14,14,PASS
Feature dictionary coverage,All eight source features,All found in feature_name,PASS


Frozen Phase 5 decisions independently validated: 31/31 checks passed.
Modeling data source: data/processed/donor_features.parquet
Exact operating threshold retained for Phase 6: 0.065423686721555


## Interpretability Setup and Phase 5 Validation

The interpretability environment was configured with consistent plotting, display, and reproducibility settings. The finalized primary and historical benchmark pipelines, feature metadata, test predictions, and evaluation artifacts were loaded directly from Phase 5 without retraining or modifying either model.

The primary model remains the `RF Moderate` Random Forest configuration using the eight feature Aggregate RFM set. Its scores remain uncalibrated and should be interpreted as relative donor ranking scores rather than literal donation probabilities. The exact operating threshold is `0.06542368672155528`, and campaign performance will continue to be examined at the 1%, 5%, 10%, and 20% outreach capacities.

All 31 validation checks passed. The saved model configuration, classifier parameters, source and transformed features, donor identifiers, target values, and development/test partitions were independently confirmed. The final test predictions, evaluation metrics, outreach results, and threshold based outreach flags were also reproduced from the saved artifacts. The historical `donor_indicator_flag` benchmark remains separate from the primary future donor target and will only be used for cohort comparison.

In [6]:
# Reconstruct aligned development and test interpretation datasets
interpretation_source_columns = [
    "donor_unique_id",
    expected_primary_decisions["target"],
    *features_primary_saved,
]

missing_interpretation_columns = set(
    interpretation_source_columns
).difference(data_modeling_full.columns)

if missing_interpretation_columns:
    raise KeyError(
        "Required interpretation columns are missing: "
        + ", ".join(sorted(missing_interpretation_columns))
    )

data_primary_interpretation_source = (
    data_modeling_full.loc[:, interpretation_source_columns]
    .copy()
)

predictions_primary_test_ordered = (
    predictions_primary_final_test.copy()
    .reset_index(drop=True)
)

predictions_primary_test_ordered[
    "_prediction_row_order"
] = np.arange(predictions_primary_test_ordered.shape[0])

data_primary_test_interpretation_merged = (
    predictions_primary_test_ordered.merge(
        data_primary_interpretation_source,
        on="donor_unique_id",
        how="left",
        validate="one_to_one",
        sort=False,
    )
    .sort_values("_prediction_row_order")
    .reset_index(drop=True)
)

test_target_artifact_matches_source = np.array_equal(
    data_primary_test_interpretation_merged[
        "target_actual_primary_donor_flag"
    ].astype(int),
    data_primary_test_interpretation_merged[
        expected_primary_decisions["target"]
    ].astype(int),
)

test_donor_id_set = set(
    data_primary_test_interpretation_merged["donor_unique_id"]
)

data_primary_development_interpretation = (
    data_primary_interpretation_source.loc[
        ~data_primary_interpretation_source[
            "donor_unique_id"
        ].isin(test_donor_id_set)
    ]
    .reset_index(drop=True)
    .copy()
)

test_prediction_columns = [
    "prob_primary_donor",
    "pred_primary_donor_flag",
    "donor_rank",
    "selected_outreach_flag",
    "dataset_partition",
]

data_primary_test_interpretation = (
    data_primary_test_interpretation_merged.loc[
        :,
        [
            "donor_unique_id",
            expected_primary_decisions["target"],
            *features_primary_saved,
            *test_prediction_columns,
        ],
    ]
    .reset_index(drop=True)
    .copy()
)

tracking_primary_development_donor_ids = (
    data_primary_development_interpretation[
        ["donor_unique_id"]
    ].copy()
)

tracking_primary_test_donor_ids = (
    data_primary_test_interpretation[
        ["donor_unique_id"]
    ].copy()
)

features_primary_development_interpretation = (
    data_primary_development_interpretation[
        features_primary_saved
    ].copy()
)

features_primary_test_interpretation = (
    data_primary_test_interpretation[
        features_primary_saved
    ].copy()
)

target_primary_development_interpretation = (
    data_primary_development_interpretation[
        expected_primary_decisions["target"]
    ].astype(int).copy()
)

target_primary_test_interpretation = (
    data_primary_test_interpretation[
        expected_primary_decisions["target"]
    ].astype(int).copy()
)

features_primary_development_transformed = (
    preprocessor_primary_model_final.transform(
        features_primary_development_interpretation
    )
)

features_primary_test_transformed = (
    preprocessor_primary_model_final.transform(
        features_primary_test_interpretation
    )
)

positive_class_index = int(
    np.flatnonzero(
        model_primary_pipeline_final.classes_ == 1
    )[0]
)

probabilities_primary_test_recalculated = (
    model_primary_pipeline_final.predict_proba(
        features_primary_test_interpretation
    )[:, positive_class_index]
)

predictions_primary_test_recalculated = (
    probabilities_primary_test_recalculated
    >= OPERATING_THRESHOLD
).astype(int)

development_donor_id_set = set(
    tracking_primary_development_donor_ids[
        "donor_unique_id"
    ]
)

test_donor_id_order_matches = np.array_equal(
    tracking_primary_test_donor_ids[
        "donor_unique_id"
    ].to_numpy(),
    predictions_primary_final_test[
        "donor_unique_id"
    ].to_numpy(),
)

partition_union_matches_source = (
    development_donor_id_set.union(test_donor_id_set)
    == set(data_modeling_full["donor_unique_id"])
)

transformed_shapes_valid = (
    features_primary_development_transformed.shape
    == (
        expected_primary_decisions["development_records"],
        expected_primary_decisions["transformed_feature_count"],
    )
    and features_primary_test_transformed.shape
    == (
        expected_primary_decisions["final_test_records"],
        expected_primary_decisions["transformed_feature_count"],
    )
)

test_ranks_valid = np.array_equal(
    np.sort(
        data_primary_test_interpretation[
            "donor_rank"
        ].astype(int).to_numpy()
    ),
    np.arange(
        1,
        data_primary_test_interpretation.shape[0] + 1,
    ),
)

interpretation_alignment_checks = {
    "Development IDs are unique and nonmissing": (
        tracking_primary_development_donor_ids[
            "donor_unique_id"
        ].notna().all()
        and tracking_primary_development_donor_ids[
            "donor_unique_id"
        ].is_unique
    ),
    "Test IDs are unique and nonmissing": (
        tracking_primary_test_donor_ids[
            "donor_unique_id"
        ].notna().all()
        and tracking_primary_test_donor_ids[
            "donor_unique_id"
        ].is_unique
    ),
    "Development and test IDs do not overlap": (
        development_donor_id_set.isdisjoint(test_donor_id_set)
    ),
    "Partitions cover the complete modeling dataset": (
        partition_union_matches_source
    ),
    "Development record count is preserved": (
        data_primary_development_interpretation.shape[0]
        == expected_primary_decisions["development_records"]
    ),
    "Test record count is preserved": (
        data_primary_test_interpretation.shape[0]
        == expected_primary_decisions["final_test_records"]
    ),
    "Test donor order matches the prediction artifact": (
        test_donor_id_order_matches
    ),
    "Test targets match the modeling dataset": (
        test_target_artifact_matches_source
    ),
    "Source feature order matches the fitted pipeline": (
        list(features_primary_development_interpretation.columns)
        == features_primary_pipeline
        and list(features_primary_test_interpretation.columns)
        == features_primary_pipeline
    ),
    "Transformed dataset shapes are correct": (
        transformed_shapes_valid
    ),
    "Saved test probabilities match the fitted pipeline": (
        np.allclose(
            probabilities_primary_test_recalculated,
            data_primary_test_interpretation[
                "prob_primary_donor"
            ].astype(float),
            rtol=1e-10,
            atol=1e-12,
        )
    ),
    "Saved test classes match the operating threshold": (
        np.array_equal(
            predictions_primary_test_recalculated,
            data_primary_test_interpretation[
                "pred_primary_donor_flag"
            ].astype(int),
        )
    ),
    "Threshold-based outreach flags remain aligned": (
        np.array_equal(
            predictions_primary_test_recalculated,
            data_primary_test_interpretation[
                "selected_outreach_flag"
            ].astype(int),
        )
    ),
    "Test donor ranks are complete and unique": (
        test_ranks_valid
    ),
}

failed_alignment_checks = [
    check_name
    for check_name, check_passed
    in interpretation_alignment_checks.items()
    if not check_passed
]

if failed_alignment_checks:
    raise AssertionError(
        "Interpretation dataset reconstruction failed for: "
        + ", ".join(failed_alignment_checks)
    )

interpretation_partition_summary = pd.DataFrame(
    [
        {
            "Partition": "Development",
            "Records": (
                data_primary_development_interpretation.shape[0]
            ),
            "Positive Targets": int(
                target_primary_development_interpretation.sum()
            ),
            "Positive Rate": (
                target_primary_development_interpretation.mean()
                * 100
            ),
            "Source Features": (
                features_primary_development_interpretation.shape[1]
            ),
            "Transformed Features": (
                features_primary_development_transformed.shape[1]
            ),
            "Saved Predictions": "Not applicable",
        },
        {
            "Partition": "Final Test",
            "Records": data_primary_test_interpretation.shape[0],
            "Positive Targets": int(
                target_primary_test_interpretation.sum()
            ),
            "Positive Rate": (
                target_primary_test_interpretation.mean()
                * 100
            ),
            "Source Features": (
                features_primary_test_interpretation.shape[1]
            ),
            "Transformed Features": (
                features_primary_test_transformed.shape[1]
            ),
            "Saved Predictions": "Aligned",
        },
    ]
)

display(
    interpretation_partition_summary.style
    .hide(axis="index")
    .set_properties(
        **{
            "text-align": "center",
            "padding": "8px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("padding", "8px"),
                ],
            }
        ]
    )
    .format(
        {
            "Records": "{:,.0f}",
            "Positive Targets": "{:,.0f}",
            "Positive Rate": "{:,.2f}%",
            "Source Features": "{:,.0f}",
            "Transformed Features": "{:,.0f}",
        }
    )
)

print(
    "Interpretation datasets reconstructed successfully: "
    f"{sum(interpretation_alignment_checks.values())}/"
    f"{len(interpretation_alignment_checks)} checks passed."
)

Partition,Records,Positive Targets,Positive Rate,Source Features,Transformed Features,Saved Predictions
Development,"27,522","1,523",5.53%,8,14,Not applicable
Final Test,"6,881",381,5.54%,8,14,Aligned


Interpretation datasets reconstructed successfully: 14/14 checks passed.


In [7]:
# Recover and label transformed primary model features
business_labels_primary_sources = {
    "feature_past_5yr_total_donation": (
        "Total Donation Amount (Past 5 Years)"
    ),
    "feature_past_5yr_average_donation": (
        "Average Donation Amount (Past 5 Years)"
    ),
    "feature_past_5yr_donation_frequency_rate": (
        "Donation Frequency Rate (Past 5 Years)"
    ),
    "feature_years_since_last_donation_past_5yr": (
        "Years Since Most Recent Donation"
    ),
    "feature_past_5yr_max_donation": (
        "Maximum Annual Donation Amount (Past 5 Years)"
    ),
    "donor_age": "Donor Age",
    "feature_gender_identity": "Gender Identity",
    "feature_preferred_address_type": "Preferred Address Type",
}

category_label_overrides = {
    "f": "Female",
    "female": "Female",
    "m": "Male",
    "male": "Male",
    "u": "Unknown",
    "unknown": "Unknown",
    "missing": "Missing",
    "nan": "Missing",
    "none": "Missing",
    "home": "Home Address",
    "business": "Business Address",
    "campus": "Campus Address",
    "other": "Other Address Type",
}


def format_category_label(category_value):
    category_text = (
        str(category_value)
        .replace("_", " ")
        .strip()
    )
    normalized_category = category_text.lower()

    return category_label_overrides.get(
        normalized_category,
        category_text.title(),
    )


def recover_source_feature(transformed_feature_name):
    transformed_name_without_prefix = (
        transformed_feature_name.split("__", 1)[-1]
    )

    ordered_source_features = sorted(
        features_primary_saved,
        key=len,
        reverse=True,
    )

    for source_feature_name in ordered_source_features:
        if transformed_name_without_prefix == source_feature_name:
            return {
                "Source Feature": source_feature_name,
                "Raw Category": None,
                "Representation": "Numeric",
            }

        categorical_prefix = f"{source_feature_name}_"

        if transformed_name_without_prefix.startswith(
            categorical_prefix
        ):
            raw_category = transformed_name_without_prefix[
                len(categorical_prefix):
            ]

            return {
                "Source Feature": source_feature_name,
                "Raw Category": raw_category,
                "Representation": "One-Hot Encoded",
            }

    return {
        "Source Feature": None,
        "Raw Category": None,
        "Representation": "Unmapped",
    }


transformed_feature_mapping_records = []

for transformed_position, transformed_feature_name in enumerate(
    features_primary_transformed,
    start=1,
):
    recovered_feature_information = recover_source_feature(
        transformed_feature_name
    )

    source_feature_name = recovered_feature_information[
        "Source Feature"
    ]
    raw_category = recovered_feature_information["Raw Category"]
    representation_type = recovered_feature_information[
        "Representation"
    ]

    source_business_label = (
        business_labels_primary_sources.get(
            source_feature_name,
            "Unmapped Feature",
        )
    )

    if raw_category is None:
        category_business_label = "Not applicable"
        transformed_business_label = source_business_label
    else:
        category_business_label = format_category_label(
            raw_category
        )
        transformed_business_label = (
            f"{source_business_label}: "
            f"{category_business_label}"
        )

    transformed_feature_mapping_records.append(
        {
            "Transformed Position": transformed_position,
            "Transformed Feature": transformed_feature_name,
            "Source Feature": source_feature_name,
            "Representation": representation_type,
            "Category Level": category_business_label,
            "Business-Friendly Label": transformed_business_label,
        }
    )

mapping_primary_transformed_features = pd.DataFrame(
    transformed_feature_mapping_records
)

source_feature_mapping_records = []

for source_feature_name in features_primary_saved:
    source_mapping_rows = (
        mapping_primary_transformed_features.loc[
            mapping_primary_transformed_features[
                "Source Feature"
            ]
            == source_feature_name
        ]
    )

    source_category_levels = source_mapping_rows.loc[
        source_mapping_rows["Representation"]
        == "One-Hot Encoded",
        "Category Level",
    ].tolist()

    source_feature_mapping_records.append(
        {
            "Source Feature": source_feature_name,
            "Business-Friendly Label": (
                business_labels_primary_sources[
                    source_feature_name
                ]
            ),
            "Representation": (
                "One-Hot Encoded"
                if source_category_levels
                else "Numeric"
            ),
            "Transformed Columns": source_mapping_rows.shape[0],
            "Category Levels": (
                ", ".join(source_category_levels)
                if source_category_levels
                else "Not applicable"
            ),
        }
    )

mapping_primary_source_features = pd.DataFrame(
    source_feature_mapping_records
)

metadata_features_available = set(
    metadata_feature_dictionary[
        metadata_feature_name_column
    ].astype(str)
)

feature_mapping_checks = {
    "All transformed features are represented": (
        mapping_primary_transformed_features.shape[0]
        == len(features_primary_transformed)
    ),
    "Transformed feature names are unique": (
        mapping_primary_transformed_features[
            "Transformed Feature"
        ].is_unique
    ),
    "Every transformed feature maps to a source feature": (
        mapping_primary_transformed_features[
            "Source Feature"
        ].notna().all()
        and (
            mapping_primary_transformed_features[
                "Representation"
            ]
            != "Unmapped"
        ).all()
    ),
    "All eight source features are represented": (
        set(
            mapping_primary_transformed_features[
                "Source Feature"
            ]
        )
        == set(features_primary_saved)
    ),
    "Source feature order is preserved": (
        mapping_primary_source_features[
            "Source Feature"
        ].tolist()
        == features_primary_saved
    ),
    "All source features have business labels": (
        set(features_primary_saved)
        == set(business_labels_primary_sources)
    ),
    "All source features exist in the feature dictionary": (
        set(features_primary_saved).issubset(
            metadata_features_available
        )
    ),
    "Development transformed width matches mapping": (
        features_primary_development_transformed.shape[1]
        == mapping_primary_transformed_features.shape[0]
    ),
    "Test transformed width matches mapping": (
        features_primary_test_transformed.shape[1]
        == mapping_primary_transformed_features.shape[0]
    ),
    "Business-friendly transformed labels are unique": (
        mapping_primary_transformed_features[
            "Business-Friendly Label"
        ].is_unique
    ),
}

failed_feature_mapping_checks = [
    check_name
    for check_name, check_passed
    in feature_mapping_checks.items()
    if not check_passed
]

if failed_feature_mapping_checks:
    raise AssertionError(
        "Transformed feature mapping failed for: "
        + ", ".join(failed_feature_mapping_checks)
    )

PRIMARY_TRANSFORMED_FEATURE_MAPPING_PATH = (
    INTERPRETABILITY_TABLES_DIR
    / "primary_transformed_feature_mapping.csv"
)

PRIMARY_SOURCE_FEATURE_MAPPING_PATH = (
    INTERPRETABILITY_TABLES_DIR
    / "primary_source_feature_mapping.csv"
)

mapping_primary_transformed_features.to_csv(
    PRIMARY_TRANSFORMED_FEATURE_MAPPING_PATH,
    index=False,
)

mapping_primary_source_features.to_csv(
    PRIMARY_SOURCE_FEATURE_MAPPING_PATH,
    index=False,
)

display(
    mapping_primary_transformed_features.style
    .hide(axis="index")
    .set_properties(
        **{
            "text-align": "center",
            "padding": "8px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("padding", "8px"),
                ],
            }
        ]
    )
    .format(
        {
            "Transformed Position": "{:,.0f}",
            "Transformed Feature": "{}",
            "Source Feature": "{}",
            "Representation": "{}",
            "Category Level": "{}",
            "Business-Friendly Label": "{}",
        }
    )
)

print("\n")

display(
    mapping_primary_source_features.style
    .hide(axis="index")
    .set_properties(
        **{
            "text-align": "center",
            "padding": "8px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("padding", "8px"),
                ],
            }
        ]
    )
    .format(
        {
            "Source Feature": "{}",
            "Business-Friendly Label": "{}",
            "Representation": "{}",
            "Transformed Columns": "{:,.0f}",
            "Category Levels": "{}",
        }
    )
)

print(
    "Transformed feature mapping completed successfully: "
    f"{sum(feature_mapping_checks.values())}/"
    f"{len(feature_mapping_checks)} checks passed."
)
print(
    "Transformed mapping saved to: "
    f"{PRIMARY_TRANSFORMED_FEATURE_MAPPING_PATH.relative_to(PROJECT_ROOT)}"
)
print(
    "Source mapping saved to: "
    f"{PRIMARY_SOURCE_FEATURE_MAPPING_PATH.relative_to(PROJECT_ROOT)}"
)

Transformed Position,Transformed Feature,Source Feature,Representation,Category Level,Business-Friendly Label
1,donor_age,donor_age,Numeric,Not applicable,Donor Age
2,feature_past_5yr_total_donation,feature_past_5yr_total_donation,Numeric,Not applicable,Total Donation Amount (Past 5 Years)
3,feature_past_5yr_average_donation,feature_past_5yr_average_donation,Numeric,Not applicable,Average Donation Amount (Past 5 Years)
4,feature_past_5yr_max_donation,feature_past_5yr_max_donation,Numeric,Not applicable,Maximum Annual Donation Amount (Past 5 Years)
5,feature_past_5yr_donation_frequency_rate,feature_past_5yr_donation_frequency_rate,Numeric,Not applicable,Donation Frequency Rate (Past 5 Years)
6,feature_years_since_last_donation_past_5yr,feature_years_since_last_donation_past_5yr,Numeric,Not applicable,Years Since Most Recent Donation
7,feature_gender_identity_Female,feature_gender_identity,One-Hot Encoded,Female,Gender Identity: Female
8,feature_gender_identity_Male,feature_gender_identity,One-Hot Encoded,Male,Gender Identity: Male
9,feature_gender_identity_Unknown,feature_gender_identity,One-Hot Encoded,Unknown,Gender Identity: Unknown
10,feature_preferred_address_type_Business,feature_preferred_address_type,One-Hot Encoded,Business Address,Preferred Address Type: Business Address


Source Feature,Business-Friendly Label,Representation,Transformed Columns,Category Levels
feature_past_5yr_total_donation,Total Donation Amount (Past 5 Years),Numeric,1,Not applicable
feature_past_5yr_average_donation,Average Donation Amount (Past 5 Years),Numeric,1,Not applicable
feature_past_5yr_donation_frequency_rate,Donation Frequency Rate (Past 5 Years),Numeric,1,Not applicable
feature_years_since_last_donation_past_5yr,Years Since Most Recent Donation,Numeric,1,Not applicable
feature_past_5yr_max_donation,Maximum Annual Donation Amount (Past 5 Years),Numeric,1,Not applicable
donor_age,Donor Age,Numeric,1,Not applicable
feature_gender_identity,Gender Identity,One-Hot Encoded,3,"Female, Male, Unknown"
feature_preferred_address_type,Preferred Address Type,One-Hot Encoded,5,"Business Address, Campus Address, Home Address, Missing, Other Address Type"


Transformed feature mapping completed successfully: 10/10 checks passed.
Transformed mapping saved to: outputs/interpretability/tables/primary_transformed_feature_mapping.csv
Source mapping saved to: outputs/interpretability/tables/primary_source_feature_mapping.csv


## Interpretation Dataset Construction and Feature Mapping

The development and final test interpretation datasets were reconstructed using the saved final test donor identifiers from Phase 5. All remaining donor records were assigned to the development partition, preserving the original split of 27,522 development records and 6,881 final test records. Donor identifiers, target values, prediction probabilities, threshold classifications, outreach selection flags, and donor ranks were all verified for correct alignment.

The development partition contains 1,523 positive targets, representing 5.53% of its records. The final test partition contains 381 positive targets, representing 5.54% of its records. All 14 reconstruction and alignment checks passed. The saved final test probabilities were also independently reproduced using the finalized pipeline.

The model uses eight source predictors that are transformed into 14 model input columns. The six numeric predictors remain as individual columns, while gender identity is expanded into three one hot encoded columns and preferred address type is expanded into five. Each transformed feature was successfully mapped back to its original source variable and assigned a clear business friendly label. The source level and transformed feature mappings were saved as reusable artifacts for the feature importance, permutation importance, and SHAP analyses.
